In [ ]:
import os 
import polars as pl
import simple_icd_10_cm as cm
from pathlib import Path

DATA_PATH = Path('/data/gusev/USERS/jpconnor/data/')
COMPASS_DATA_PATH = DATA_PATH / 'CAIA/COMPASS/'
PROFILE_DATA_PATH = DATA_PATH / 'PROFILE_DATA/'
CLIN_EMBED_PATH = DATA_PATH / 'clinical_text_embedding_project/'

mrns = pl.read_csv(COMPASS_DATA_PATH / 'mrn_lists/icd_prostate_mrn_flags.csv')

somatic_df = pl.read_parquet(PROFILE_DATA_PATH / 'SOMATIC_WIDE_BY_SAMPLE.parquet')
genomic_spec_df = (pl.scan_parquet(PROFILE_DATA_PATH / 'GENOMIC_SPECIMEN.parquet')
                   .filter(pl.col('TEST_TYPE') != 'RAPIDHEME_CLINICAL')
                   .collect(engine='streaming'))
meds_summary = pl.read_parquet(PROFILE_DATA_PATH / 'MEDICATIONS_SUMMARY.parquet')
careg_df = pl.read_parquet(PROFILE_DATA_PATH / 'CAREG.parquet')
ehr_df = pl.read_parquet(PROFILE_DATA_PATH / 'EHR_DIAGNOSES.parquet')

old_somatic_df = pl.read_csv(CLIN_EMBED_PATH / 'clinical_and_genomic_features/complete_somatic_data_df.csv.gz')
old_stage_df = pl.read_csv(CLIN_EMBED_PATH / 'clinical_and_genomic_features/cancer_stage_df.csv.gz')
old_type_df = pl.read_csv(CLIN_EMBED_PATH / 'clinical_and_genomic_features/cancer_type_df.csv.gz')
text_meta = pl.read_csv(CLIN_EMBED_PATH / 'batched_datasets/processed_datasets/full_VTE_embeddings_metadata.csv.gz')

### Process ICD data

In [ ]:
diagnosis_cols = [
    "DIAGNOSIS_ICD10_CD",
    "DIAGNOSIS_ICD10_NM",
    "DIAGNOSIS_ICD10_CD2",
    "DIAGNOSIS_ICD10_NM2",
    "DIAGNOSIS_ICD10_CD3",
    "DIAGNOSIS_ICD10_NM3",
]

ICD_COL = 'DIAGNOSIS_ICD10_CD'

df_long = (
    ehr_df
    .with_columns(
        diagnosis=pl.concat_list(
            [
                pl.struct(
                    pl.col("DIAGNOSIS_ICD10_CD").alias("CD"),
                    pl.col("DIAGNOSIS_ICD10_NM").alias("NM"),
                ),
                pl.struct(
                    pl.col("DIAGNOSIS_ICD10_CD2").alias("CD"),
                    pl.col("DIAGNOSIS_ICD10_NM2").alias("NM"),
                ),
                pl.struct(
                    pl.col("DIAGNOSIS_ICD10_CD3").alias("CD"),
                    pl.col("DIAGNOSIS_ICD10_NM3").alias("NM"),
                ),
            ]
        )
    )
    .drop(diagnosis_cols)
    .explode("diagnosis", empty_as_null=True)
    .unnest("diagnosis")
    .rename(
        {
            "CD": "DIAGNOSIS_ICD10_CD",
            "NM": "DIAGNOSIS_ICD10_NM",
        }
    )
    .filter(
        pl.col("DIAGNOSIS_ICD10_CD")
        .fill_null("")
        .str.strip_chars()
        .ne("")
    )
)

deduped_ICD_df = df_long[['DFCI_MRN', 'START_DT', 'DIAGNOSIS_ICD10_CD']].drop_nulls().unique()

def lookup_icd(code: str | None) -> dict:
    
    blank_template = {
        ICD_COL : code, 
        "DIAGNOSIS_ICD10_NM" : None,
        "ICD10_CATEGORY_CODE" : None,
        "ICD10_CATEGORY_NAME" : None,
    }
    
    code = code.strip().upper()
    
    if not cm.is_valid_item(code):
        return blank_template
    
    category_code = code.split('.')[0][:3]
    group_code = cm.get_ancestors(category_code)[0]

    updated_template = blank_template.copy()
    updated_template['DIAGNOSIS_ICD10_NM'] = cm.get_description(code)
    updated_template['ICD10_CATEGORY_CODE'] = category_code
    updated_template['ICD10_CATEGORY_NAME'] = cm.get_description(category_code)

    return updated_template

deduped_ICD_df = deduped_ICD_df.with_columns(
    pl.col(ICD_COL)
    .cast(pl.String)
    .str.strip_chars()
    .str.to_uppercase()
)

unique_codes = (
    deduped_ICD_df.select(ICD_COL)
    .drop_nulls()
    .unique()
    .get_column(ICD_COL)
    .to_list()
)

icd_map = pl.DataFrame(
    [lookup_icd(code) for code in unique_codes],
    schema={
        ICD_COL : pl.String,
        'DIAGNOSIS_ICD10_NM' : pl.String,
        'ICD10_CATEGORY_CODE' : pl.String,
        'ICD10_CATEGORY_NAME' : pl.String,
    }
)

unique_groups = icd_map[['ICD10_CATEGORY_CODE', 'ICD10_CATEGORY_NAME']].unique().sort(['ICD10_CATEGORY_CODE', 'ICD10_CATEGORY_NAME'], descending=[False, False])

malignant_groups = icd_map.filter(
    pl.col('ICD10_CATEGORY_CODE').str.contains(r'^C')
)
metastatic_groups = malignant_groups.filter(
    pl.col('ICD10_CATEGORY_CODE').is_in(['C77', 'C78', 'C79'])
)

malignant_ICDs = malignant_groups['DIAGNOSIS_ICD10_CD'].unique().to_list()
metastatic_ICDs = metastatic_groups['DIAGNOSIS_ICD10_CD'].unique().to_list()

deduped_ICD_df = deduped_ICD_df.with_columns(
    pl.col(ICD_COL).is_in(malignant_ICDs)
    .alias('MALIGNANT_CODE')
)
deduped_ICD_df = deduped_ICD_df.with_columns(
    pl.col(ICD_COL).is_in(metastatic_ICDs)
    .alias('METASTATIC_CODE')
)

deduped_malignant_df = (
    deduped_ICD_df
    .with_columns(
        pl.col("START_DT").cast(pl.Date, strict=False)
    )
    .filter(
        pl.col("MALIGNANT_CODE").fill_null(False)
        | pl.col("METASTATIC_CODE").fill_null(False)
    )
    .group_by(
        [
            "DFCI_MRN",
            "DIAGNOSIS_ICD10_CD",
            "MALIGNANT_CODE",
            "METASTATIC_CODE",
        ]
    )
    .agg(
        pl.col("START_DT").min().alias("START_DT")
    )
    .sort(
        ["DFCI_MRN", "START_DT", "DIAGNOSIS_ICD10_CD"]
    )
    .join(icd_map, on=ICD_COL)
)

### CAREG analysis

In [ ]:
deduped_malignant_df['DFCI_MRN'].unique()

In [ ]:
# filter would be 
# ICD defined diagnosis date
# meds summary defined treatment date
# text available

In [ ]:
meds_summary['DFCI_MRN'].unique()

In [ ]:
careg_mrns = careg_df.filter((~pl.col('BEST_AJCC_STAGE_CD').is_in(['99', '88'])) &
                             (pl.col('BEST_AJCC_STAGE_CD').is_not_null()))['DFCI_MRN'].unique()
meds_summary_mrns = meds_summary.filter((pl.col('MED_NCI_PREFERRED_NM_1').is_not_null()) &
                                        (pl.col('MED_START_DT_1').is_not_null()))['DFCI_MRN'].unique()
text_mrns = text_meta['DFCI_MRN'].unique()

In [ ]:
surv_df = pl.read_csv(COMPASS_DATA_PATH / 'survival_analysis/prediction_inputs_adt/aggregated_landmark0.csv')
prostate_sample = genomic_spec_df.filter(pl.col('DFCI_MRN').is_in(surv_df['DFCI_MRN']) & 
                                         (pl.col('CANCER_TYPE') == 'Prostate Cancer'))
non_prostate_samples = genomic_spec_df.filter((pl.col('DFCI_MRN').is_in(surv_df['DFCI_MRN'])) &
                                              (pl.col('CANCER_TYPE') != 'Prostate Cancer'))

In [ ]:
#genomics based info for prostate cancer
# spec df --> loc CANCER_TYPE == 'Prostate Cancer', MRNs intersect with cohort
# take sequencing date as earliest of ['SAMPLE_COLLECTION_DT', 'REPORT_DT', 'TEST_ORDER_DT']
# take sequencing closest to ADT initiation date
# use SAMPLE_ACCESSION_NBR in SOMATIC_WIDE_BY_SAMPLE.parquet as merge condition
# use _SV, _SNV, _AMP, _DEL as indicators of binary genomic data